# Oracle search algorithms: Deutsch-Jozsa

This notebook starts a pair of algorithms that share a central idea: the oracle, a function encoded as a quantum gate that can be queried in superposition. Deutsch-Jozsa is the simpler of the two, and [`05b_grover.ipynb`](05b_grover.ipynb), Grover, reuses the same idea.

## The problem

There is a function `f` from `n` bits to 1 bit, guaranteed to be either **constant**, same output for every input, or **balanced**, half the inputs give 0 and the other half give 1. The goal is to decide which of the two it is, querying `f` as few times as possible.

Classically, in the worst case, more than half the inputs need to be queried to be certain. Deutsch-Jozsa decides it with a single query, thanks to being able to evaluate `f` on a superposition of all inputs at once.

## What an oracle is

An oracle is the function `f` encoded as a reversible quantum gate: `|x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩`. Register `x` is the input qubits, and `y` is an auxiliary qubit that accumulates the result with an XOR.

By preparing the auxiliary qubit in the `|−⟩` state before querying the oracle, the result of `f(x)` shows up as a phase change on the input register instead of being added to the auxiliary qubit. This trick, called phase kickback, is what allows extracting information from all inputs at once.

## The algorithm

1. `n` input qubits in `|0⟩`, one auxiliary qubit in `|1⟩`.
2. H on every qubit, including the auxiliary one: this leaves it in `|−⟩`.
3. Apply the oracle.
4. H again on the input qubits, leaving the auxiliary one alone.
5. Measure the input qubits: all zeros means `f` is constant, any other result means `f` is balanced.

## A constant oracle

The simplest function, `f(x) = 0` for every input, needs no gate at all: the auxiliary qubit never changes.

In [ ]:
import polypus

constant_qc = polypus.Circuit(3)
constant_qc.x(2)  # auxiliary qubit in |1>
constant_qc.h(0)
constant_qc.h(1)
constant_qc.h(2)
# oracle: f(x) = 0, no gates
constant_qc.h(0)
constant_qc.h(1)
constant_qc.measure(0, 0)
constant_qc.measure(1, 1)

result = polypus.run_quantum_circuit(constant_qc, shots=1000, infrastructure="local")
print(result.counts[0])

The result is `'00'` 100% of the time: the signature of a constant function.

## A balanced oracle

`f(x0, x1) = x0 ⊕ x1` is balanced: of the four possible inputs, two give 0 and two give 1. It is implemented with a CX from each input qubit to the auxiliary one:

In [ ]:
balanced_qc = polypus.Circuit(3)
balanced_qc.x(2)
balanced_qc.h(0)
balanced_qc.h(1)
balanced_qc.h(2)
balanced_qc.cx(0, 2)  # oracle: f(x0, x1) = x0 xor x1
balanced_qc.cx(1, 2)
balanced_qc.h(0)
balanced_qc.h(1)
balanced_qc.measure(0, 0)
balanced_qc.measure(1, 1)

With the pattern seen in [`02b`](02b_qiskit_qasm_interop.ipynb), the full circuit:

In [ ]:
from qiskit import qasm2

qc_drawing = qasm2.loads(
    balanced_qc.to_qasm2(), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS
)
qc_drawing.draw("mpl")

In [ ]:
result = polypus.run_quantum_circuit(balanced_qc, shots=1000, infrastructure="local")
print(result.counts[0])

The result is `'11'` 100% of the time, never `'00'`: the signature of a balanced function. Telling the two apart took a single query to the oracle, not several.

## Summary

This notebook introduced the oracle as a quantum function queryable in superposition, and Deutsch-Jozsa as the simplest algorithm that takes advantage of it: it tells a constant function apart from a balanced one with a single query, where classically several are needed.

[`05b_grover.ipynb`](05b_grover.ipynb), Grover, reuses the same oracle concept to search an unsorted list.

## Next step

Continue with: [`05b_grover.ipynb`](05b_grover.ipynb).